In [9]:
from bs4 import BeautifulSoup
import json

def load_nq_dataset(file_path):
    data = []
    with open (file_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            html_content = record.get("document_text", "")

            # Parse HTML and extract text
            soup = BeautifulSoup(html_content, "html.parser")
            plain_text = soup.get_text(separator = " ", strip = True)
            data.append(plain_text)
    return data



In [10]:
load_nq_dataset(r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\simplified-nq-train.jsonl")

MemoryError: 

In [12]:
print (data)

NameError: name 'data' is not defined

In [1]:
from bs4 import BeautifulSoup
import json

def stream_nq_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            html_content = record.get("document_text", "")
            soup = BeautifulSoup(html_content, "html.parser")
            plain_text = soup.get_text(separator=" ", strip=True)
            yield plain_text  # Use yield instead of appending to a list

In [ ]:
# Example usage: process line-by-line
for i, text in enumerate(stream_nq_dataset(r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\simplified-nq-train.jsonl")):
    if i < 5:  # Just print first 5 for sanity check
        print(text[:200], "\n---\n")

In [2]:
with open("cleaned_output.txt", "w", encoding="utf-8") as out:
    for text in stream_nq_dataset(r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\simplified-nq-train.jsonl"):
        out.write(text + "\n")

In [4]:
# chunking the data

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

def stream_and_chunk(path, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    with open(path, 'r', encoding='utf-8') as f:
        buffer = ""
        for line in f:
            buffer += line
            if len(buffer) >= chunk_size:
                doc = Document(page_content=buffer)
                for chunk in text_splitter.split_documents([doc]):
                    yield chunk
                buffer = ""
        # Handle leftover text
        if buffer:
            doc = Document(page_content=buffer)
            for chunk in text_splitter.split_documents([doc]):
                yield chunk

# Example: print first 5 chunks
for i, chunk in enumerate(stream_and_chunk(r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\cleaned_output.txt")):
    if i < 5:
        print(f"Chunk {i+1}:\n{chunk.page_content[:300]}\n---\n")

c:\Users\kanna\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunk 1:
Email marketing - Wikipedia Email marketing Jump to : navigation , search ( hide ) This article has multiple issues . Please help improve it or discuss these issues on the talk page . ( Learn how and when to remove these template messages ) This article needs additional citations for verification . 
---

Chunk 2:
this template message ) Part of a series on Internet marketing Search engine optimization Local search engine optimisation Social media marketing Email marketing Referral marketing Content marketing Native advertising Search engine marketing Pay - per - click Cost per impression Search analytics Web
---

Chunk 3:
meant to build loyalty , trust , or brand awareness . Marketing emails can be sent to a purchased lead list or a current customer database . The term usually refers to sending email messages with the purpose of enhancing a merchant 's relationship with current or previous customers , encouraging cus
---

Chunk 4:
rapidly alongside the technological growth of 

In [ ]:
# Collect all chunks into a list
documents = list(stream_and_chunk(r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\cleaned_output.txt"))



In [6]:
!pip install sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Load a BERT-based model (you can change this to any Hugging Face model)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"  # Lightweight and fast
)

# Create vector store
db = Chroma.from_documents(documents, embedding_model)

: 